In [1]:
from pathlib import Path
import duckdb
import pandas as pd

class DataHubReader:
    def __init__(self, db_name="yahoo_finance.db"):
        """
        Stellt Verbindung zur DuckDB her.
        Erwartet dieselbe Ordnerstruktur wie DataHub.
        """
        base_path = Path.cwd()
        db_path = base_path.parent.parent / "data" / "01_raw" / "yahoo" / db_name

        if not db_path.exists():
            raise FileNotFoundError(f"Datenbank nicht gefunden: {db_path}")

        self.con = duckdb.connect(str(db_path))

    def _fetch_table(self, table_name: str) -> pd.DataFrame:
        """Generische Methode zum Laden einer Tabelle."""
        return self.con.execute(f"SELECT * FROM {table_name}").df()

    def get_financials(self) -> pd.DataFrame:
        """Yahoo Finance Financials"""
        return self._fetch_table("bronze_financials")

    def get_wikidata(self) -> pd.DataFrame:
        """Wikidata Unternehmensdaten"""
        return self._fetch_table("bronze_wikidata")

    def get_GMD(self) -> pd.DataFrame:
        """Global Macro Database"""
        return self._fetch_table("bronze_gmd")

    def get_interest(self) -> pd.DataFrame:
        """IMF Zinsdaten"""
        return self._fetch_table("bronze_interest")

    def get_commodity(self) -> pd.DataFrame:
        """IMF Commodity Daten"""
        return self._fetch_table("bronze_commodities")

    def close(self):
        """Schließt die Datenbankverbindung."""
        self.con.close()

In [13]:
df = DataHubReader().get_wikidata()

In [14]:
df

,ticker,company_qid,item_description,value,ingested_at
0,ALATA.PA,Q753684,isin,FR0010478248,2026-03-26 13:59:18.278280
1,ALATA.PA,Q753684,isin,US4659401040,2026-03-26 13:59:18.278280
2,ALCGM.PA,Q1052675,isin,FR0000053506,2026-03-26 13:59:18.278280
3,ALECP.PA,Q1375196,isin,FR0010490920,2026-03-26 13:59:18.278280
4,ALCLS.PA,Q2943995,isin,FR0010425595,2026-03-26 13:59:18.278280
...,...,...,...,...,...
36639,TGYM.MI,Q2461960,owned_by,nan,2026-03-26 13:59:18.278280
36640,ZEST.MI,Q93097534,owned_by,nan,2026-03-26 13:59:18.278280
36641,TPRO.MI,Q111142424,owned_by,Q112479322,2026-03-26 13:59:18.278280
36642,ZUC.MI,Q115167588,owned_by,nan,2026-03-26 13:59:18.278280


In [31]:
df['item_description'].value_counts()

item_description
instance_of       5522
subsidiaries      5156
investments       4315
products          3035
industries        3012
owned_by          2807
part_of           2698
operating_area    2658
isin              2532
founding_year     2522
location          2387
Name: count, dtype: int64

In [44]:
from datetime import datetime
import numpy as np
class WikidatatoSilver():
    def __init__(self):
        pass
    def correct_nan(self,df):
        df = df.copy()
        df.loc[df['value'] == 'nan', 'value'] = np.nan
        return df
    def get_age(self, df):
        fj = df[df['item_description'] == 'founding_year'].drop_duplicates(subset=["company_qid"]).copy()
        current_year = datetime.now().year
        fj["value"] = current_year - pd.to_datetime(fj["value"], errors="coerce").dt.year 
        fj['item_description'] = 'company_age'
        return fj
    def replace_qids_with_ticker(self, df):
        df = df.copy()
        mapping = (
            df[["company_qid", "ticker"]]
            .dropna(subset=["company_qid", "ticker"])
            .drop_duplicates(subset=["company_qid", "ticker"])
            .drop_duplicates(subset=["company_qid"], keep="first")
            .set_index("company_qid")["ticker"]
        )
        mask = df["value"].astype(str).str.match(r"^Q\d+$", na=False)
        df.loc[mask, "value"] = (
            df.loc[mask, "value"].astype(str).str.strip()
            .map(mapping)
            .fillna(df.loc[mask, "value"])
        )
        return df
    def run(self, df):
        nan = self.correct_nan(df)
        nan = nan.dropna()
        with_tickers = self.replace_qids_with_ticker(nan)
        return with_tickers
    

    

In [48]:
pd.set_option("display.max_rows", None)
def classify_value_type(df):
    df = df.copy()
    s = df["value"].astype("string").str.strip()

    is_qid = s.str.fullmatch(r"Q\d+", na=False)
    is_isin = s.str.fullmatch(r"[A-Z]{2}[A-Z0-9]{9}\d", na=False)
    is_date = pd.to_datetime(s, errors="coerce", utc=True).notna()

    df["value_type"] = "ticker"

    df.loc[s.isna() | (s == ""), "value_type"] = "missing"
    df.loc[is_qid, "value_type"] = "qid"
    df.loc[is_isin, "value_type"] = "isin"
    df.loc[is_date, "value_type"] = "date"

    return df
erg = WikidatatoSilver().run(df)
s = erg["value"].astype("string").str.strip()

is_qid = s.str.fullmatch(r"Q\d+", na=False)
is_isin = s.str.fullmatch(r"[A-Z]{2}[A-Z0-9]{9}\d", na=False)
is_date = pd.to_datetime(s, errors="coerce").notna()

is_ticker = ~(is_qid | is_isin | is_date) & s.notna() & (s != "")

print("Ticker-Kandidaten:", is_ticker.sum())


C:\Users\Konra\AppData\Local\Temp\ipykernel_8932\1446774560.py:23: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  is_date = pd.to_datetime(s, errors="coerce").notna()


Ticker-Kandidaten: 342
